# 模型評估圖表

**這份 notebook 只負責呼叫既有繪圖腳本並顯示結果，不含任何繪圖邏輯。**
繪圖邏輯全部留在 `_plot_common.py` / `plot_*.py` 裡，之後要改圖表樣式、加新指標，只需要改那些 `.py` 檔案，這份 notebook 不用跟著動。

**資料來源**：只讀 `results/` 底下的檔案，不重新跑模型、不碰 `test.csv`。

| 圖表 | 需要的檔案 | 由誰產生 |
|---|---|---|
| Per-class recall | `results/predictions.parquet` | `build_predictions.py` |
| F1 / Recall over rounds | `results/metrics_rounds.csv` | 組員手動整理，格式見 `results/metrics_rounds_TEMPLATE.csv` |
| ROC curves | `results/preds_*.csv`（可以有多份，一份一個情境） | 組員手動整理 |

缺哪一份，對應的圖表區塊會顯示清楚的提示，不會噴錯。

## Setup

確認工作目錄在 `our-project/analyze/`（Jupyter 預設用開啟 notebook 所在的目錄當 cwd，如果不是，下面會印出警告），並列出 `results/` 底下目前有哪些檔案，一眼看出還缺什麼。

In [ ]:
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

CWD = Path.cwd()
print(f"目前工作目錄: {CWD}")

_expected_scripts = [
    "_plot_common.py", "plot_f1.py", "plot_recall.py",
    "plot_roc.py", "plot_per_class_recall.py",
]
_missing_scripts = [f for f in _expected_scripts if not (CWD / f).exists()]
if _missing_scripts:
    print(
        f"[警告] 目前工作目錄下找不到 {_missing_scripts}，"
        "這份 notebook 預期在 our-project/analyze/ 底下開啟執行，請確認 Jupyter 的啟動目錄。"
    )
else:
    print("[Info] 工作目錄正確，找得到所有繪圖腳本。")

if str(CWD) not in sys.path:
    sys.path.insert(0, str(CWD))

RESULTS_DIR = CWD / "results"
FIGURES_DIR = CWD / "figures"

print("\nresults/ 底下目前有的檔案:")
if RESULTS_DIR.exists():
    for p in sorted(RESULTS_DIR.iterdir()):
        print(f"  {p.name}")
else:
    print("  (results/ 資料夾不存在)")

## Per-Class Recall

每種 `attack_raw` 類型被判成攻擊的比例（門檻 0.5）。`observe` 那一條是假陽性率（越低越好），其餘是 recall（越高越好）——圖裡用顏色區分，數值表格也在下面。

資料來源：`results/predictions.parquet`（由 `build_predictions.py` 產生）。

In [ ]:
from plot_per_class_recall import main as plot_per_class_recall_main

_per_class_table = plot_per_class_recall_main()

_img_path = FIGURES_DIR / "per_class_recall.png"
if _img_path.exists() and _per_class_table is not None:
    display(Image(filename=str(_img_path)))
    display(_per_class_table)
else:
    display(Markdown(
        "**[缺資料]** 找不到 `results/predictions.parquet`，這張圖需要先跑 "
        "`build_predictions.py` 產生預測快取。"
    ))

## F1 Score over Rounds

各 scenario 的 F1 隨 FL round 的變化。資料來源：`results/metrics_rounds.csv`。

In [ ]:
from plot_f1 import main as plot_f1_main

plot_f1_main()

_img_path = FIGURES_DIR / "f1_by_round.png"
if _img_path.exists():
    display(Image(filename=str(_img_path)))
else:
    display(Markdown(
        "**[等待資料]** 還沒有 `results/metrics_rounds.csv`，"
        "等待組員提供資料，格式見 `results/metrics_rounds_TEMPLATE.csv`。"
    ))

## Recall over Rounds

各 scenario 的 Recall 隨 FL round 的變化。資料來源同上，`results/metrics_rounds.csv`。

In [ ]:
from plot_recall import main as plot_recall_main

plot_recall_main()

_img_path = FIGURES_DIR / "recall_by_round.png"
if _img_path.exists():
    display(Image(filename=str(_img_path)))
else:
    display(Markdown(
        "**[等待資料]** 還沒有 `results/metrics_rounds.csv`，"
        "等待組員提供資料，格式見 `results/metrics_rounds_TEMPLATE.csv`。"
    ))

## ROC Curves

每個 `results/preds_<scenario>.csv` 畫一條 ROC 曲線疊在同一張圖上比較，圖例標出各自的 AUC。

In [ ]:
from plot_roc import main as plot_roc_main

plot_roc_main()

_img_path = FIGURES_DIR / "roc_curve.png"
if _img_path.exists():
    display(Image(filename=str(_img_path)))
else:
    display(Markdown(
        "**[等待資料]** 還沒有 `results/preds_*.csv`，"
        "等待組員提供資料（格式見下方說明）。"
    ))

## 資料格式需求

**`results/metrics_rounds.csv`**（表頭範本見 `results/metrics_rounds_TEMPLATE.csv`）：

| 欄位 | 說明 |
|---|---|
| `scenario` | 自由字串，例如 `baseline` / `fixed_criterion` / `true_bagging`，有幾個就畫幾條線 |
| `round` | FL round 編號，整數，從 1 開始 |
| `accuracy` / `precision` / `recall` / `f1` / `auc` | 該 scenario 在該 round 的指標，浮點數 0-1 |

每個 `(scenario, round)` 組合一列。同一個 scenario 只有一個 round（例如都只填 `round=1`）時，折線圖會自動改畫長條圖。

**`results/preds_<scenario>.csv`**：

| 欄位 | 說明 |
|---|---|
| `y_true` | 0 或 1（0=observe，1=攻擊） |
| `y_score` | **模型輸出的機率**，0 到 1 之間的浮點數，不是門檻化之後的 0/1 predict——ROC 曲線需要看不同門檻下的變化，全部填 0/1 的話畫出來只會是一個點，不是一條曲線 |

檔名裡 `preds_` 後面的部分會直接當成圖例上的 scenario 名稱，例如 `preds_bagging_v2.csv` 會顯示成 `bagging_v2`。